In [ ]:
from pathlib import Path
import os
import sys
import json

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from depth_anything_3.api import DepthAnything3

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.common.geometry.pointcloud import (
    transform_lidar_to_ego,
    transform_ego_to_global
)
from src.common.geometry.depth import transform_cam_to_ego
from src.common.geometry.transform import make_transform, invert_transform
from src.common.visualize.pointcloud import plot_pointcloud
from src.common.visualize.depth import plot_depth_with_original_image, plot_pseudo_lidar_with_ground_truth

from src.depth_anything3.inference import get_pseudo_lidar

# Resolve paths relative to this notebook directory
MODEL_NAME = "DA3METRIC-LARGE"
device = "cuda" if torch.cuda.is_available() else "cpu"
# Create the model and load the weights
model = DepthAnything3.from_pretrained(f"depth-anything/{MODEL_NAME}")
model = model.to(device=device)
# Load nuScenes dataset
NUSCENES_ROOT = Path.cwd().parent / "data/nuscenes"
NUSCENES_VERSION = "v1.0-trainval"
with open(NUSCENES_ROOT / NUSCENES_VERSION / "scene.json") as f:
    scenes = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample.json") as f:
    samples_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_data.json") as f:
    sample_data_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "ego_pose.json") as f:
    ego_poses_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "calibrated_sensor.json") as f:
    calibrated_sensors_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sensor.json") as f:
    sensors = json.load(f)

# Create hash maps for token lookup
sensors = {sensor["token"]: sensor for sensor in sensors}
sensor_lookup = {sensor["channel"]: sensor["token"] for sensor in sensors.values()}

print(f"sensor_channels: {[sensor['channel'] for sensor in sensors.values()]}")
print(f"scene_names: {[scene['name'] for scene in scenes]}")

In [ ]:
# Select the scenes and camera channel
SCENE_NAME = "scene-0061"
CAMERA_CHANNEL = "CAM_FRONT"
lidar_channel = "LIDAR_TOP"

scene = next(scene for scene in scenes if scene["name"] == SCENE_NAME)
# Select the samples
samples = [sample for sample in samples_all if sample["scene_token"] == scene["token"]]
print(f"num_samples: {len(samples)}")
sample_tokens = set(sample["token"] for sample in samples)
# Select the sample_data, ego_poses, and calibrated_sensors in the scene
sample_data = {sd["token"]: sd for sd in sample_data_all if sd["sample_token"] in sample_tokens}
sample_data = {sd_token: sd for sd_token, sd in sample_data.items() if sd["is_key_frame"]}  # Filter by is_key_frame
sample_data = dict(sorted(sample_data.items(), key=lambda item: item[1]["timestamp"]))  # sort by timestamp
ego_pose_tokens = set(sd["ego_pose_token"] for sd in sample_data.values())
ego_poses = {ep["token"]: ep for ep in ego_poses_all if ep["token"] in ego_pose_tokens}
calibrated_sensor_tokens = set(sd["calibrated_sensor_token"] for sd in sample_data.values())
calibrated_sensors = {cs["token"]: cs for cs in calibrated_sensors_all if cs["token"] in calibrated_sensor_tokens}

# Filter sample_data to only include the selected camera channel
calibrated_sensors_cam = {cs_token: cs for cs_token, cs in calibrated_sensors.items() if cs["sensor_token"] == sensor_lookup[CAMERA_CHANNEL]}
sample_data_cam = {sd_token: sd for sd_token, sd in sample_data.items() if sd["calibrated_sensor_token"] in calibrated_sensors_cam}
ego_poses_cam = {ep_token: ep for ep_token, ep in ego_poses.items() if ep_token in set(sd["ego_pose_token"] for sd in sample_data_cam.values())}

calibrated_sensors_lidar = {cs_token: cs for cs_token, cs in calibrated_sensors.items() if cs["sensor_token"] == sensor_lookup[lidar_channel]}
sample_data_lidar = {sd_token: sd for sd_token, sd in sample_data.items() if sd["calibrated_sensor_token"] in calibrated_sensors_lidar}
ego_poses_lidar = {ep_token: ep for ep_token, ep in ego_poses.items() if ep_token in set(sd["ego_pose_token"] for sd in sample_data_lidar.values())}


# Show the first three sample data entries for the selected scene
for i, (token, sd) in enumerate(sample_data_cam.items()):
    if i >= 3:
        break
    # Read and display the image
    image_path = NUSCENES_ROOT / sd["filename"]
    image = Image.open(image_path)
    plt.imshow(image)
    plt.axis("off")
    plt.show()
    # Read the point cloud
    lidar_sd = next(sdl for sdl in sample_data_lidar.values() if sdl["sample_token"] == sd["sample_token"])
    lidar_path = NUSCENES_ROOT / lidar_sd["filename"]
    lidar_points = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
    points_lidar = lidar_points[:, :3]  # Extract x, y, z coordinates
    intensity = lidar_points[:, 3]  # Extract intensity values
    # Transform the points from the LiDAR frame to the global frame
    lidar_calibration = calibrated_sensors_lidar[lidar_sd["calibrated_sensor_token"]]
    lidar_points_ego = transform_lidar_to_ego(points_lidar, 
                                              lidar_translation=lidar_calibration["translation"],
                                              lidar_quaternion=lidar_calibration["rotation"])
    ego_pose = ego_poses_lidar[lidar_sd["ego_pose_token"]]
    lidar_points_global = transform_ego_to_global(lidar_points_ego,
                                                  ego_translation=ego_pose["translation"],
                                                  ego_quaternion=ego_pose["rotation"])
    # Visualize the point cloud using Open3D
    fig = plot_pointcloud(lidar_points_global,
                          axis_translation=ego_pose["translation"],
                          axis_quaternion=ego_pose["rotation"])
    fig.show()
    

In [ ]:
# Depth estimation by DepthAnything3
# Show the first three sample data entries for the selected scene
for i, (token, sd) in enumerate(sample_data_cam.items()):
    if i >= 3:
        break
    # Read the image
    image_path = NUSCENES_ROOT / sd["filename"]
    image = Image.open(image_path)
    # Get the camera parameters
    camera_translation = calibrated_sensors_cam[sd["calibrated_sensor_token"]]["translation"]
    camera_rotation = calibrated_sensors_cam[sd["calibrated_sensor_token"]]["rotation"]
    camera_intrinsic = calibrated_sensors_cam[sd["calibrated_sensor_token"]]["camera_intrinsic"]
    # Read the point cloud
    lidar_sd = next(sdl for sdl in sample_data_lidar.values() if sdl["sample_token"] == sd["sample_token"])
    lidar_path = NUSCENES_ROOT / lidar_sd["filename"]
    lidar_points = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
    points_lidar = lidar_points[:, :3]  # Extract x, y, z coordinates
    intensity = lidar_points[:, 3]  # Extract intensity values
    # Transform the points from the LiDAR frame to the global frame
    lidar_calibration = calibrated_sensors_lidar[lidar_sd["calibrated_sensor_token"]]
    lidar_points_ego = transform_lidar_to_ego(points_lidar, 
                                              lidar_translation=lidar_calibration["translation"],
                                              lidar_quaternion=lidar_calibration["rotation"])
    ego_pose = ego_poses_lidar[lidar_sd["ego_pose_token"]]
    lidar_points_global = transform_ego_to_global(lidar_points_ego,
                                                  ego_translation=ego_pose["translation"],
                                                  ego_quaternion=ego_pose["rotation"])
    # Inference depth using DepthAnything3 (without pose conditioning)
    prediction = model.inference([image])
    depth_map = prediction.depth[0]
    # Show the depth map
    plot_depth_with_original_image(depth_map, np.array(image))
    # Generate a point cloud by pseudo-lidar from the depth map and transform to global coordinates
    pseudo_lidar_points = get_pseudo_lidar(depth_map, 
                                           np.array(camera_intrinsic),
                                           original_image_width=image.width,
                                           original_image_height=image.height,
                                           model_name="DA3METRIC-LARGE")
    pseudo_points_ego = transform_cam_to_ego(pseudo_lidar_points,
                                             camera_translation=camera_translation,
                                             camera_quaternion=camera_rotation)
    fig = plot_pointcloud(pseudo_points_ego)
    fig.show()
    pseudo_points_global = transform_ego_to_global(pseudo_points_ego,
                                                   ego_translation=ego_pose["translation"],
                                                   ego_quaternion=ego_pose["rotation"])
    # Visualize the point cloud using Open3D
    fig = plot_pseudo_lidar_with_ground_truth(
        pseudo_lidar_points=pseudo_points_global,
        ground_truth_points=lidar_points_global,
        axis_translation=ego_pose["translation"],
        axis_quaternion=ego_pose["rotation"]
    )
    fig.show()

In [ ]:
# Pose-conditioned depth estimation with multiple samples
# Use camera intrinsics and extrinsics for pose-conditioned depth estimation
intrinsics = np.array(camera_intrinsic)[None, ...]
camera_to_ego = make_transform(quaternion=camera_rotation, translation=camera_translation)
extrinsics = invert_transform(camera_to_ego)[None, ...]  # DA3 expects world (ego in this case) to camera extrinsics
prediction = model.inference([image], extrinsics=extrinsics, intrinsics=intrinsics)